# 🔮 PredictiveOps — Live Demo

**Cloud-native predictive reliability platform.**  
Ingests real-time telemetry → scores infrastructure risk → auto-heals before outages occur.

---

### What this notebook demos

| Scenario | What happens |
|---|---|
| **Gradual degradation** | Latency creeps up, error rate follows, risk crosses 0.75, auto-heal fires, metrics recover |
| **Sudden spike** | Instant max-risk event, immediate heal |
| **All 4 runbooks** | Cycles restart-appservice, failover-storage, reroute-network, clear-socket-connections |
| **Chaos mode** | Random unpredictable signals, watch the risk score react in real time |

> **No Azure account needed.** The risk engine and auto-heal logic run fully locally inside this notebook.  
> GitHub: [github.com/Gvld3iii/predictiveops-test](https://github.com/Gvld3iii/predictiveops-test)

---

**Run all cells in order**, or jump straight to a scenario at the bottom.

In [ ]:
# ── Cell 1: Install deps ───────────────────────────────────────────────────
# Nothing to install — pure Python stdlib only.
import time, random, math, textwrap
from datetime import datetime, timezone
from IPython.display import display, HTML, clear_output

print('✅  Dependencies ready.')

In [ ]:
# ── Cell 2: Risk engine (mirrors production RiskEngine/__init__.py) ────────

RISK_THRESHOLD = 0.75

def compute_risk(latency_ms: float, error_rate_pct: float, nxdomain: bool) -> float:
    """Weighted composite risk score — identical to production logic."""
    risk = 0.0
    if latency_ms    >= 200: risk += 0.45
    if error_rate_pct >= 1.0: risk += 0.45
    if nxdomain:              risk += 0.15
    return min(1.0, round(risk, 2))

def auto_heal(resource_id: str, risk: float, runbook: str = 'restart-appservice') -> dict:
    """Simulates AutoHeal function — executes runbook and returns confirmation."""
    time.sleep(0.3)
    return {
        'ok': True,
        'action': runbook,
        'resourceId': resource_id,
        'riskAtTrigger': risk,
        'healed': True,
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }

def process_telemetry(resource_id, latency, error_rate, nxdomain, runbook='restart-appservice'):
    """Full pipeline: score → persist → heal if needed."""
    risk   = compute_risk(latency, error_rate, nxdomain)
    healed = None
    if risk >= RISK_THRESHOLD:
        healed = auto_heal(resource_id, risk, runbook)
    return {
        'resourceId':        resource_id,
        'latency':           latency,
        'errorRate':         error_rate,
        'nxdomainAnomaly':   nxdomain,
        'risk':              risk,
        'autoHealTriggered': healed is not None,
        'healResult':        healed,
        'timestamp':         datetime.now(timezone.utc).isoformat(),
    }

print('✅  Risk engine loaded.')
print(f'    Threshold: {RISK_THRESHOLD}')
print(f'    Signals:   latency ≥200ms (+0.45) · error rate ≥1% (+0.45) · NXDOMAIN (+0.15)')

In [ ]:
# ── Cell 3: Display helpers ────────────────────────────────────────────────

def risk_bar(risk: float, width: int = 24) -> str:
    filled = round(risk * width)
    return '█' * filled + '░' * (width - filled)

def risk_colour(risk: float) -> str:
    if risk >= 0.75: return '\033[91m'
    if risk >= 0.45: return '\033[93m'
    return '\033[92m'

RESET = '\033[0m'
BOLD  = '\033[1m'
GREY  = '\033[90m'
CYAN  = '\033[96m'
GREEN = '\033[92m'
RED   = '\033[91m'

def print_event(result: dict, label: str = '') -> None:
    r    = result['risk']
    rc   = risk_colour(r)
    tag  = f"{GREY}[{label}]{RESET} " if label else ''
    heal = f"  {GREEN}{BOLD}🔧 AUTO-HEAL FIRED → {result['healResult']['action']}{RESET}" if result['autoHealTriggered'] else ''
    print(
        f"  {tag}"
        f"latency={CYAN}{result['latency']:>6.1f}ms{RESET}  "
        f"err={CYAN}{result['errorRate']:>5.2f}%{RESET}  "
        f"nx={CYAN}{str(result['nxdomainAnomaly']):<5}{RESET}  "
        f"risk={rc}{r:.2f}{RESET} {rc}{risk_bar(r, 20)}{RESET}"
        f"{heal}"
    )

def section(title: str) -> None:
    print(f"\n{BOLD}{'─'*60}")
    print(f"  {title}")
    print(f"{'─'*60}{RESET}\n")

print('✅  Display helpers loaded.')

---
## Scenario 1 — Gradual Degradation

The most realistic demo. Latency climbs slowly, error rate follows, the system detects the pattern **before a full outage**, crosses the threshold, and auto-heals. Watch the risk bar fill up in real time.

In [ ]:
section('SCENARIO 1 — GRADUAL DEGRADATION → AUTO-HEAL')

GRADUAL_STEPS = [
    ('healthy baseline',       85.0,   0.05,  False),
    ('normal load',           110.0,   0.08,  False),
    ('slight slowdown',       145.0,   0.12,  False),
    ('degradation begins',    175.0,   0.30,  False),
    ('latency spike',         210.0,   0.55,  False),
    ('errors climbing',       240.0,   0.90,  False),
    ('dns anomaly detected',  255.0,   1.10,  True ),
    ('CRITICAL — heal fires', 280.0,   1.60,  True ),  # ← heal fires here
    ('post-heal recovery',     95.0,   0.10,  False),
    ('back to baseline',       80.0,   0.04,  False),
]

resource = 'demo-app-service-prod'
history  = []

for label, latency, error_rate, nxdomain in GRADUAL_STEPS:
    result = process_telemetry(resource, latency, error_rate, nxdomain)
    history.append(result)
    print_event(result, label)
    time.sleep(1.2)

print(f"\n  {GREEN}{BOLD}Demo complete.{RESET}")
print(f"  {GREY}Peak risk: {max(r['risk'] for r in history):.2f}  |  Heals: {sum(r['autoHealTriggered'] for r in history)}{RESET}")

---
## Scenario 2 — Sudden Spike

Instant max-risk event. Heal fires immediately, one recovery event confirms the system is back.

In [ ]:
section('SCENARIO 2 — SUDDEN SPIKE → IMMEDIATE HEAL')

SPIKE_STEPS = [
    ('pre-spike baseline', 90.0,   0.05, False),
    ('SPIKE',             350.0,   2.50, True ),
    ('post-heal',          88.0,   0.04, False),
]

resource = 'prod-api-gateway'

for label, latency, error_rate, nxdomain in SPIKE_STEPS:
    result = process_telemetry(resource, latency, error_rate, nxdomain)
    print_event(result, label)
    time.sleep(1.0)

print(f"\n  {GREEN}{BOLD}Done.{RESET}")

---
## Scenario 3 — All 4 Runbooks

Demonstrates the full breadth of auto-remediation. Each resource type triggers a different PowerShell runbook.

In [ ]:
section('SCENARIO 3 — ALL 4 RUNBOOK SCENARIOS')

RUNBOOK_SCENARIOS = [
    {
        'runbook':     'restart-appservice',
        'resource':    'prod-api-app-service',
        'description': 'App service memory leak — high latency + errors',
        'latency':     260.0, 'error_rate': 1.80, 'nxdomain': False,
    },
    {
        'runbook':     'failover-storage',
        'resource':    'prod-storage-account-east',
        'description': 'Storage account unresponsive — timeouts spiking',
        'latency':     310.0, 'error_rate': 2.10, 'nxdomain': False,
    },
    {
        'runbook':     'reroute-network',
        'resource':    'prod-vnet-gateway',
        'description': 'Network gateway degraded — DNS anomaly + latency',
        'latency':     230.0, 'error_rate': 1.20, 'nxdomain': True,
    },
    {
        'runbook':     'clear-socket-connections',
        'resource':    'prod-load-balancer',
        'description': 'Load balancer socket exhaustion — error rate critical',
        'latency':     195.0, 'error_rate': 1.50, 'nxdomain': False,
    },
]

for i, sc in enumerate(RUNBOOK_SCENARIOS, 1):
    print(f"  {BOLD}Scenario {i}/4 — {sc['runbook']}{RESET}")
    print(f"  {GREY}{sc['description']}{RESET}")

    baseline = process_telemetry(sc['resource'], 85.0, 0.05, False, sc['runbook'])
    print_event(baseline, 'baseline')
    time.sleep(0.8)

    trigger = process_telemetry(sc['resource'], sc['latency'], sc['error_rate'], sc['nxdomain'], sc['runbook'])
    print_event(trigger, f"trigger → {sc['runbook']}")
    time.sleep(0.8)

    recovery = process_telemetry(sc['resource'], 82.0, 0.03, False, sc['runbook'])
    print_event(recovery, 'recovered')

    print()
    if i < len(RUNBOOK_SCENARIOS):
        time.sleep(1.0)

print(f"  {GREEN}{BOLD}All 4 runbook scenarios complete.{RESET}")

---
## Scenario 4 — Chaos Mode

Unpredictable random signals. Shows the system handling real-world noise — mostly healthy with occasional degradation and spikes. Runs for 20 events.

In [ ]:
section('SCENARIO 4 — CHAOS MODE')

resource   = 'prod-chaos-target'
heal_count = 0

for i in range(1, 21):
    roll = random.random()
    if roll < 0.55:
        latency, error_rate, nxdomain = random.uniform(60, 160),  random.uniform(0, 0.4),  False
    elif roll < 0.80:
        latency, error_rate, nxdomain = random.uniform(160, 260), random.uniform(0.4, 1.2), random.random() < 0.2
    else:
        latency, error_rate, nxdomain = random.uniform(260, 400), random.uniform(1.0, 3.0), random.random() < 0.5

    result = process_telemetry(resource, latency, error_rate, nxdomain)
    if result['autoHealTriggered']:
        heal_count += 1
    print_event(result, f'#{i:02d}')
    time.sleep(0.6)

print(f"\n  {GREEN}{BOLD}Chaos complete.{RESET}  {GREY}Heals triggered: {heal_count}/20{RESET}")

---
## Risk Score Explorer

Test any telemetry values manually and see exactly how the scoring model responds.

In [ ]:
# ── Adjust these values and re-run the cell ────────────────────────────────

LATENCY_MS    = 220.0   # milliseconds
ERROR_RATE    = 0.8     # percent (1.0 = 1%)
NXDOMAIN      = False   # DNS anomaly detected?
RESOURCE_ID   = 'my-custom-resource'

# ──────────────────────────────────────────────────────────────────────────

result = process_telemetry(RESOURCE_ID, LATENCY_MS, ERROR_RATE, NXDOMAIN)

rc = risk_colour(result['risk'])
print(f"\n  Resource  : {CYAN}{RESOURCE_ID}{RESET}")
print(f"  Latency   : {CYAN}{LATENCY_MS}ms{RESET}  {'→ +0.45' if LATENCY_MS >= 200 else '→ no penalty'}")
print(f"  Error rate: {CYAN}{ERROR_RATE}%{RESET}   {'→ +0.45' if ERROR_RATE >= 1.0 else '→ no penalty'}")
print(f"  NXDOMAIN  : {CYAN}{NXDOMAIN}{RESET}   {'→ +0.15' if NXDOMAIN else '→ no penalty'}")
print(f"  ────────────────────────────────")
print(f"  Risk score: {rc}{BOLD}{result['risk']:.2f}{RESET} {rc}{risk_bar(result['risk'])}{RESET}")
print(f"  Threshold : {RISK_THRESHOLD}")
print(f"  Action    : {GREEN + BOLD + '✓ healthy' if not result['autoHealTriggered'] else RED + BOLD + '🔧 AUTO-HEAL TRIGGERED'}{RESET}")

---

## Summary

| Component | Role |
|---|---|
| `RiskEngine` | Azure Function — receives telemetry, scores risk, triggers heal |
| `AutoHeal` | Azure Function — executes PowerShell runbooks via webhook |
| `RiskStream` | Azure Function — streams events to the dashboard |
| `Cosmos DB` | Persistent audit log of all risk events |
| `shared_state.py` | Thread-safe in-memory event ring buffer |
| Dashboard | Live HTML/JS observability UI |

**Stack:** Python 3.x · Azure Functions v2 · Azure Cosmos DB · PowerShell Automation Runbooks · Vanilla JS dashboard

---

🔗 **GitHub:** [github.com/Gvld3iii/predictiveops-test](https://github.com/Gvld3iii/predictiveops-test)